In [ ]:
from openai import OpenAI
import pandas as pd
import json
import time

# -----------------------------
# LLM Client Setup
# -----------------------------
client = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")
model_name = "qwen/qwen3-vl-30b"

# -----------------------------
# File Paths
# -----------------------------
INPUT_FILE = "Illegal_Trade_News_Article.csv"
OUTPUT_FILE = "Illegal_Trade_News_Article_with_countries.csv"

# -----------------------------
# JSON Schema for extraction
# -----------------------------
country_schema = {
    "type": "json_schema",
    "json_schema": {
        "name": "BuyerSellerCountryExtraction",
        "schema": {
            "type": "object",
            "properties": {
                "buyer_country": {
                    "type": "string",
                    "description": "Country of the buyer/importer/destination. Use 'Unknown' if not stated."
                },
                "seller_country": {
                    "type": "string",
                    "description": "Country of the seller/exporter/origin. Use 'Unknown' if not stated."
                },
                "evidence": {
                    "type": "string",
                    "description": "Brief quote or summary from the text supporting the decision."
                }
            },
            "required": ["buyer_country", "seller_country", "evidence"]
        },
    }
}

# -----------------------------
# Load input CSV
# -----------------------------
df = pd.read_csv(INPUT_FILE, dtype=str).fillna("")

# Ensure column exists
if "main_content" not in df.columns:
    raise ValueError("Column 'main_content' not found in the CSV.")

# Output columns
df["buyer_country"] = ""
df["seller_country"] = ""
df["evidence"] = ""

# -----------------------------
# Loop through articles
# -----------------------------
for i, row in df.iterrows():
    start_time = time.time()

    content = row["main_content"].strip()

    # If empty content, skip safely
    if not content:
        df.at[i, "buyer_country"] = "Unknown"
        df.at[i, "seller_country"] = "Unknown"
        df.at[i, "evidence"] = "main_content was empty."
        continue

    prompt_text = f"""
You extract buyer and seller countries from news articles about (illegal) trade.

Definitions:
- buyer_country = the country that is buying/importing/receiving the goods (destination market).
- seller_country = the country that is selling/exporting/supplying the goods (origin/source).

Rules:
- Use ONLY country names (e.g., "United States", "China", "Brazil").
- If multiple are mentioned, choose the primary buyer and primary seller most clearly tied to the trade transaction described.
- If the article only implies a direction without naming a country, output "Unknown".
- If locations are cities/regions, infer the country ONLY if unambiguous (e.g., "Paris" -> "France"). Otherwise "Unknown".
- If the text is about enforcement but doesn't clearly specify buyer/seller countries, return "Unknown" for whichever is missing.

Return JSON only.

Article:
{content}
""".strip()

    response = client.chat.completions.create(
        model=model_name,
        messages=[{
            "role": "user",
            "content": [{"type": "text", "text": prompt_text}]
        }],
        response_format=country_schema
    )

    raw = response.choices[0].message.content
    try:
        parsed = json.loads(raw) if isinstance(raw, str) else raw
        buyer = parsed.get("buyer_country", "Unknown")
        seller = parsed.get("seller_country", "Unknown")
        evidence = parsed.get("evidence", "")
    except Exception:
        buyer, seller, evidence = "Unknown", "Unknown", f"Failed to parse model output: {raw}"

    df.at[i, "buyer_country"] = buyer
    df.at[i, "seller_country"] = seller
    df.at[i, "evidence"] = evidence

    print(f"\n📰 Article {i+1} of {len(df)}")
    print(f"🛒 Buyer: {buyer}")
    print(f"📦 Seller: {seller}")
    # print(f"⏱️ Runtime: {round(time.time() - start_time, 2)} seconds")

# -----------------------------
# Save output
# -----------------------------
df.to_csv(OUTPUT_FILE, index=False)
print(f"\n✅ Done. Saved to {OUTPUT_FILE}")